# Heston calibration & Monte Carlo — period August 2008 – July 2009

**Period file:** **2008-08-01 → 2009-07-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** choose lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM exercise decision on SPY American calls (risk-neutral paths from the same simulator); results in `stopping_results`.

**Estimation:** V3 option-implied calibration of \(\kappa,\theta,\xi,\rho,v_0\) from listed call prices (Fourier / characteristic function). Monte Carlo and LSM unchanged.

True rolling rule: at each update, re-calibrate \(\hat\kappa,\hat\theta,\hat\xi,\hat\rho,\hat v_0\) to listed calls in the current lookback and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.



## 0. Setup



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
PERIOD_START = pd.Timestamp("2008-08-01")
PERIOD_END = pd.Timestamp("2009-07-31")
TICKERS = ["AAPL", "MSFT", "SPY"]
N_DAYS = 252
N_STEPS = 500  # Monte Carlo time steps per path
MIN_WINDOW = 60  # unused by option-implied Heston block; kept for return-window helpers
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "3 months": pd.DateOffset(months=3),
    "6 months": pd.DateOffset(months=6),
    "1 year": pd.DateOffset(years=1),
    "2 years": pd.DateOffset(years=2),
    "3 years": pd.DateOffset(years=3),
    "5 years": pd.DateOffset(years=5),
}
ROLLING_OPTIONS = ["daily", "monthly", "none"]

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

rolling = {}
cal_meta = {}

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()



## 1. Stock price trends (August 2008 – July 2009)

Adjusted close for AAPL, MSFT, and SPY (primary).



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, August 2008 – July 2009")
axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period August 2008 – July 2009", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))



## 2. Strike prices in this period

Unique strikes \(K\) from `*_options_panel.csv` with `trading_date` in **2008-08-01 → 2009-07-31**.

> AAPL strikes are on the option/contract scale; equity adj closes are split-adjusted.



In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)



## 3. Estimation formulas (Heston — P from returns, Q via Pan 2002)

Physical \(P\) (Method A on lookback returns):

$$dS_t/S_t=\mu\,dt+\sqrt{v_t}\,dW^{P,S},\qquad dv_t=\kappa(\theta-v_t)\,dt+\xi\sqrt{v_t}\,dW^{P,v}$$

Volatility risk is **not** identified from returns (variance is not a traded asset). Pan (2002) eqs. (2.5) and Appendix D: premium \(\eta_v\) from listed calls, \(\xi,\rho\) held at \(P\),

$$\kappa^*=\kappa-\eta_v,\qquad \bar v^*=\kappa\bar v/\kappa^*,\qquad dv=[\kappa(\bar v-v)+\eta_v v]\,dt+\xi\sqrt{v}\,dW^{Q,v}$$

$$dS_t/S_t=r_f\,dt+\sqrt{v_t}\,dW^{Q,S}$$

| Object | Under \(P\) | Risk premium | Under \(Q\) |
|--------|-------------|--------------|-------------|
| \(\mu\) | lookback mean | equity (Girsanov) | \(r_f\) |
| \(\kappa,\theta\) | Method A | \(\eta_v\) from options | \(\kappa-\eta_v\), \(\kappa\theta/\kappa^*\) |
| \(\xi,\rho\) | Method A | — | unchanged |
| \(v_0\) | recent RV | — | option-implied \(v_0^Q\) |

§4 graphs: \(P\) parameters. §5 Monte Carlo uses **\(P\)**. §6 LSM uses **\(Q\)** only.




## 4. Calibration only — August 2008 – July 2009

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.



In [ ]:
import sys
from pathlib import Path as _CalPath
_CAL_SCRIPTS = str((_CalPath("..") / "scripts").resolve())
if _CAL_SCRIPTS not in sys.path:
    sys.path.insert(0, _CAL_SCRIPTS)
from pq_risk_premium import (
    estimate_heston_p,
    calibrate_heston_q,
    report_heston_pq,
    simulate_heston_q,
)
from heston_option_calibration import (
    load_calls_panel,
    select_quotes_asof,
    calibrate_heston_from_quotes,
    physical_mu,
    merton_jump_params,
    quotes_fingerprint,
)


def _heston_opt_dir():
    if "OPT_DIR" in globals():
        return Path(OPT_DIR)
    data = Path(DATA)
    if int(N_DAYS) > 400:
        return data / "options" / "processed" / "short_interval"
    return data / "options" / "processed"

def estimate_heston_params(log_rets: pd.Series, quotes=None, x0=None):
    """Option-implied (κ, θ, ξ, ρ, v0); μ from lookback stock returns (P-measure)."""
    mu = physical_mu(log_rets, N_DAYS)
    n = int(log_rets.dropna().shape[0])
    cal = calibrate_heston_from_quotes(quotes, x0=x0, max_nfev=50)
    if not cal["success"] or not np.isfinite(cal["kappa"]):
        return mu, np.nan, np.nan, np.nan, np.nan, np.nan, n
    return mu, cal["kappa"], cal["theta"], cal["xi"], cal["rho"], cal["v0"], n

def _slice_window(rets: pd.Series, end: pd.Timestamp, offset: pd.DateOffset) -> pd.Series:
    start = end - offset
    return rets.loc[(rets.index > start) & (rets.index <= end)]


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    if rolling_mode == "daily":
        update_dates = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    elif rolling_mode == "monthly":
        t0 = period_prices[ticker].dropna().index[0]
        month_ends = pd.date_range(PERIOD_START, PERIOD_END, freq="ME")
        update_dates = pd.DatetimeIndex([t0]).append(month_ends).unique().sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_prices[ticker].dropna().index[0]])

    panel = load_calls_panel(Path(DATA), ticker, opt_dir=_heston_opt_dir())
    prev_q = None
    prev_fp = None
    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        quotes = select_quotes_asof(panel, pd.Timestamp(t_u), offset)
        fp = quotes_fingerprint(quotes)
        p = estimate_heston_p(window, N_DAYS)
        if not p["success"]:
            continue
        if prev_q is not None and fp == prev_fp and fp:
            q = prev_q
        else:
            q = calibrate_heston_q(p, quotes)
            if q["success"]:
                prev_q, prev_fp = q, fp
            elif prev_q is not None:
                q = prev_q
        mu = p["mu"] if np.isfinite(p["mu"]) else 0.0
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min() if len(window) else pd.Timestamp(t_u),
            "window_end": window.index.max() if len(window) else pd.Timestamp(t_u),
            "n_days": p["n"],
            "mu": mu,
            "kappa": p["kappa"],
            "theta": p["theta"],
            "xi": p["xi"],
            "rho": p["rho"],
            "v0": p["v0"],
            "eta_v": q.get("eta_v", np.nan),
            "kappa_q": q.get("kappa_q", np.nan),
            "theta_q": q.get("theta_q", np.nan),
            "v0_q": q.get("v0_q", np.nan),
            "n_quotes": q.get("n_quotes", 0),
            "q_success": bool(q.get("success", False)),
            "q_explosive": bool(q.get("q_explosive", False)),
        })

    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced trading-day grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]


def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()

    cols = ["mu", "kappa", "theta", "xi", "rho", "v0"]
    arrs = {c: cal[c].to_numpy(dtype=float) for c in cols}
    steps = {c: np.empty(n_steps, dtype=float) for c in cols}

    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        for c in cols:
            steps[c][i] = arrs[c][idx]

    return (
        dates,
        steps["mu"], steps["kappa"], steps["theta"], steps["xi"], steps["rho"],
        steps["v0"],
        float(hist.iloc[0]), hist,
    )


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Six rolling-parameter graphs: μ, θ, κ, ξ, ρ, v0."""
    panels = [
        ("mu", "μ̂ (annual)", "Estimated drift"),
        ("theta", "θ̂ (var)", "Long-run variance"),
        ("kappa", "κ̂", "Mean-reversion speed"),
        ("xi", "ξ̂", "Vol-of-vol"),
        ("rho", "ρ̂", "Price–vol correlation"),
        ("v0", "v̂₀ (var)", "Initial variance"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(6, 1, figsize=(11, 14), sharex=True)
        for ax, (col, ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = pd.to_datetime(r["date"])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[col], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            if col in {"mu", "rho"}:
                ax.axhline(0, color="0.5", lw=0.8)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)


# Reset kernel-side UI handles so reopen + Run All cannot reuse stale widgets
plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 years",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="monthly",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (graphs only)</b> — lookback + rolling, then <b>Reestimate</b>. "
        "Option-implied: κ̂, θ̂, ξ̂, ρ̂, v̂₀ from listed calls (Fourier). μ̂ from lookback returns."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated (P + Pan Q):** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)

        p_rows, prem_rows, q_rows = [], [], []
        for t in TICKERS:
            if rolling[t] is None or len(rolling[t]) == 0:
                continue
            last = rolling[t].iloc[-1]
            p_tbl, prem, q_tbl = report_heston_pq(last)
            p_tbl["ticker"] = t
            prem["ticker"] = t
            q_tbl["ticker"] = t
            p_rows.append(p_tbl)
            prem_rows.append(prem)
            q_rows.append(q_tbl)
        if p_rows:
            display(Markdown("### Physical-measure (P) parameters"))
            display(pd.DataFrame(p_rows))
            display(Markdown("### Volatility risk premium (Pan $\\eta_v$, from listed calls)"))
            display(pd.DataFrame(prem_rows))
            display(Markdown("### Risk-neutral (Q) dynamics"))
            display(pd.DataFrame(q_rows))
        display(Markdown(
            "Go to **§5** and click **Start** for one MC pair per company. "
            "Also estimated in the same windows (for simulation): "
            r"$v_0$ initializes each path; $v_t$ then follows the Heston SDE."
        ))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()




## 5. Monte Carlo only — one graph pair per company (August 2008 – July 2009)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.



In [ ]:
def simulate_heston_rolling(
    mu_step, kappa_step, theta_step, xi_step, rho_step, v0_step,
    S0, n_paths, seed,
):
    """Heston Euler MC (no jumps); params may change by step (rolling schedule)."""
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    v = np.full(n_paths, float(v0_step[0]), dtype=float)

    for i in range(n_steps):
        mu = mu_step[i]
        kappa = kappa_step[i]
        theta = theta_step[i]
        xi = xi_step[i]
        rho = float(np.clip(rho_step[i], -0.999, 0.999))

        z_v = rng.standard_normal(n_paths)
        z_indep = rng.standard_normal(n_paths)
        z_s = rho * z_v + np.sqrt(max(1.0 - rho**2, 0.0)) * z_indep

        v_pos = np.maximum(v, 0.0)
        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * v_pos) * dt
            + np.sqrt(v_pos * dt) * z_s
        )
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * z_v
        v = np.maximum(v, 0.0)
    return paths


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        (
            dates_now, mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            S0_now, hist_now,
        ) = param_schedule_for_steps(ticker, rolling[ticker])
        paths = simulate_heston_rolling(
            mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            S0_now, n_paths, seed,
        )
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            x = dates_now[:_n]
            axes[0].plot(x, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Heston Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(x, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(x, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(x, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("date")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(100.0 * np.sqrt(np.mean(((p50 - _hist) / np.maximum(np.abs(_hist), 1e-8)) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')} | option-implied",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · "
            f"**ICP (25–75)** = `{100*icp:.1f}%` · "
            f"**avg band width** = `{abw:.4f}` · "
            f"RMSE%(p50) = `{rmse:.2f}%` | seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)



## 6. Optimal stopping (American calls — Heston)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Heston simulator** and §4 calibration on **SPY / AAPL / MSFT** to decide exercise vs wait for American calls.

**Do not average simulated stock paths before stopping decisions.** Generate a cloud of individual risk-neutral paths (each keeps its own shocks). Then Longstaff–Schwartz:

1. At each exercise date, compute the immediate payoff $\max(S_t-K,0)$ **on every path**.
2. Estimate continuation by regression on in-the-money simulated states ($1, S, S^2$, path vol proxy).
3. **Each path** exercises iff payoff $>$ continuation; otherwise it continues.
4. Discount that path's stopping payoff to $t=0$.
5. The American value is the **average of those discounted payoffs** (then $\max$ with the $t=0$ intrinsic).

Paths for pricing are **$Q$** (Pan volatility risk premium; Heston $P$ from returns). §5 is $P$ visualization only. §5's expected-vs-history plot is visualization only — it is not the input to LSM.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping** (all three underlyings).




In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from pq_risk_premium import simulate_heston_q, report_heston_pq
from american_lsm import (
    STOP_TICKERS,
    lsm_american_call,
    load_calls,
    params_asof,
    sample_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Pan Q-Heston paths (not μ → r on P variance)."""
    ticker = str(getattr(row, "underlying", "SPY")).upper()
    if ticker not in rolling or len(rolling[ticker]) == 0:
        raise RuntimeError(f"No {ticker} calibration — run Reestimate in §4 first.")
    p = params_asof(rolling[ticker], row.trading_date)
    if p is None or not np.isfinite(p.get("eta_v", np.nan)):
        raise RuntimeError(f"No Q premium for {ticker} — need option quotes in §4.")
    dte = int(row.dte)
    if dte < 2:
        raise ValueError("dte must be >= 2")
    r = float(row.r)
    S0 = float(row.S_t)
    n = dte
    v0q = float(p["v0_q"]) if np.isfinite(p.get("v0_q", np.nan)) else float(p["v0"])
    steps = {
        "rf": np.full(n, r, dtype=float),
        "kappa": np.full(n, float(p["kappa"]), dtype=float),
        "theta": np.full(n, float(p["theta"]), dtype=float),
        "xi": np.full(n, float(p["xi"]), dtype=float),
        "rho": np.full(n, float(p["rho"]), dtype=float),
        "v0": np.full(n, v0q, dtype=float),
        "v0_q": np.full(n, v0q, dtype=float),
        "eta_v": np.full(n, float(p["eta_v"]), dtype=float),
    }
    return simulate_heston_q(steps, S0, n_paths, seed, n_days=int(N_DAYS) if int(N_DAYS) <= 400 else 252)



_STOP_TICKERS = list(STOP_TICKERS)
_contracts_by_ticker = {}
for _t in _STOP_TICKERS:
    _panel = load_calls(DATA, _t)
    _contracts_by_ticker[_t] = sample_calls(
        _panel, PERIOD_START, PERIOD_END, 
    )

stopping_results = {}  # ticker -> DataFrame

for _t in _STOP_TICKERS:
    _n = len(_contracts_by_ticker[_t])
    display(Markdown(
        f"Sampled **{_n}** {_t} American calls in "
        f"{PERIOD_START.date()} → {PERIOD_END.date()} "
        f"(one nearest-ATM call each Monday, or the next session if Monday is closed; DTE 7–60)."
    ))
    if _n:
        display(
            _contracts_by_ticker[_t][
                ["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]
            ].head(8)
        )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            missing = [t for t in _STOP_TICKERS if t not in rolling or len(rolling[t]) == 0]
            if missing:
                display(Markdown(
                    "Run **Reestimate** in §4 first "
                    f"(need calibration for: {', '.join(missing)})."
                ))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            dt = 1.0 / 252.0  # trading-day clock for LSM; never the 1-min N_DAYS
            stopping_results = {}

            for ticker in _STOP_TICKERS:
                contracts = _contracts_by_ticker[ticker]
                if contracts is None or len(contracts) == 0:
                    display(Markdown(f"No {ticker} call contracts in this period panel slice."))
                    continue

                rows = []
                example = None
                for i, row in enumerate(contracts.itertuples(index=False)):
                    paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                    res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                    err = res.price - float(row.option_price)
                    rows.append({
                        "ticker": ticker,
                        "trading_date": row.trading_date,
                        "S_t": float(row.S_t),
                        "K": float(row.K),
                        "dte": int(row.dte),
                        "r": float(row.r),
                        "market": float(row.option_price),
                        "model_price": res.price,
                        "error": err,
                        "early_ex_frac": res.early_exercise_frac,
                        "mean_ex_day": res.mean_exercise_step,
                    })
                    if example is None:
                        example = (row, paths, res)

                df = pd.DataFrame(rows)
                stopping_results[ticker] = df
                rmse = float(100.0 * np.sqrt(np.mean((df["error"] / np.maximum(np.abs(df["market"]), 1e-8)) ** 2)))
                mae = float(np.mean(np.abs(df["error"])))
                color = COLORS.get(ticker, "#2ca02c")

                display(Markdown(
                    f"### Heston — LSM results ({ticker})\n"
                    f"n_paths={n_paths} | contracts={len(df)} | "
                    f"RMSE={rmse:.2f}% | MAE={mae:.4f} | "
                    f"mean early-exercise fraction="
                    f"{df['early_ex_frac'].mean():.3f}"
                ))
                display(
                    df[
                        ["trading_date", "S_t", "K", "dte", "market", "model_price",
                         "error", "early_ex_frac", "mean_ex_day"]
                    ].round(4)
                )

                with plt.ioff():
                    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                    ax = axes[0]
                    ax.scatter(df["market"], df["model_price"], alpha=0.75, color=color)
                    lo = min(df["market"].min(), df["model_price"].min())
                    hi = max(df["market"].max(), df["model_price"].max())
                    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                    ax.set_xlabel("market option_price")
                    ax.set_ylabel("model LSM price")
                    ax.set_title("Price: model vs market")

                    axes[1].bar(
                        ["model", "market"],
                        [df["model_price"].mean(), df["market"].mean()],
                        color=[color, "#7f7f7f"],
                    )
                    axes[1].set_title("Mean option value")
                    axes[1].set_ylabel("price")

                    axes[2].hist(
                        df["mean_ex_day"], bins=12,
                        color=color, alpha=0.85, edgecolor="white",
                    )
                    axes[2].set_xlabel("mean exercise day (by contract)")
                    axes[2].set_title("Optimal exercise timing")
                    fig.suptitle(
                        f"Heston optimal stopping | {ticker} | "
                        f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')}",
                        fontsize=11, y=1.02,
                    )
                    fig.tight_layout()
                _show_fig(fig)

                if example is not None:
                    row, paths, res = example
                    j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                    t_ex = int(res.exercise_steps[j])
                    with plt.ioff():
                        fig2, ax = plt.subplots(figsize=(10, 3.8))
                        ax.plot(paths[j], color=color, lw=1.5, label="one RN path")
                        ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                        ax.scatter(
                            [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                            label=f"exercise day {t_ex}",
                        )
                        ax.set_xlabel("day")
                        ax.set_ylabel("S")
                        ax.set_title(
                            f"{ticker} example path | "
                            f"trade {pd.Timestamp(row.trading_date).date()} | "
                            f"dte={int(row.dte)} | model={res.price:.3f} vs "
                            f"mkt={float(row.option_price):.3f}"
                        )
                        ax.legend(frameon=False, loc="best")
                        fig2.tight_layout()
                    _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(dict keyed by ticker → model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY / AAPL / MSFT American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))





## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY / AAPL / MSFT calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.


